In [2]:
import os
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

MODEL_NAME = "GroNLP/hateBERT"

# 固定三分类标签映射
id2label = {
    0: "normal",
    1: "offensive",
    2: "hatespeech"
}
label2id = {v: k for k, v in id2label.items()}


def load_csv_as_hf_dataset(path, text_column, label_column="label"):
    df = pd.read_csv(path, encoding="utf-8")

    if text_column not in df.columns:
        raise ValueError(f"{path} 中找不到文本列: {text_column}")
    if label_column not in df.columns:
        raise ValueError(f"{path} 中找不到标签列: {label_column}")

    df = df[[text_column, label_column]].dropna().copy()
    df[text_column] = df[text_column].astype(str)
    df[label_column] = df[label_column].astype(int)

    # 统一列名
    df = df.rename(columns={text_column: "text", label_column: "label"})
    return Dataset.from_pandas(df, preserve_index=False), df


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    precision, recall, _, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "macro_precision": precision,
        "macro_recall": recall,
    }


def train_one_dataset(
    train_path,
    val_path,
    test_path,
    text_column,
    output_dir,
    label_column="label",
    max_length=128,
    learning_rate=2e-5,
    train_batch_size=16,
    eval_batch_size=32,
    epochs=5,
    weight_decay=0.01,
    seed=42
):
    print("开始训练")
    os.makedirs(output_dir, exist_ok=True)

    # 1. load data
    train_ds, train_df = load_csv_as_hf_dataset(train_path, text_column, label_column)
    val_ds, val_df = load_csv_as_hf_dataset(val_path, text_column, label_column)
    test_ds, test_df = load_csv_as_hf_dataset(test_path, text_column, label_column)

    dataset = DatasetDict({
        "train": train_ds,
        "validation": val_ds,
        "test": test_ds
    })

    print("\n===== Data Summary =====")
    print("Train size:", len(train_df))
    print(train_df["label"].value_counts().sort_index())
    print("\nVal size:", len(val_df))
    print(val_df["label"].value_counts().sort_index())
    print("\nTest size:", len(test_df))
    print(test_df["label"].value_counts().sort_index())

    # 2. tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            max_length=max_length
        )

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 3. model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=3,
        id2label=id2label,
        label2id=label2id
    )

    # 4. training args
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        num_train_epochs=3,
        weight_decay=weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        seed=seed,
        report_to="none"
    )

    # 5. trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # 6. train
    trainer.train()

    best_dir = os.path.join(output_dir, "best_model")
    trainer.save_model(best_dir)
    tokenizer.save_pretrained(best_dir)

    # 7. evaluation
    print("\n===== Validation Result =====")
    val_result = trainer.evaluate(tokenized_dataset["validation"])
    print(val_result)

    print("\n===== Test Result =====")
    test_result = trainer.evaluate(tokenized_dataset["test"])
    print(test_result)

    # 8. detailed report
    pred_output = trainer.predict(tokenized_dataset["test"])
    preds = np.argmax(pred_output.predictions, axis=-1)
    labels = pred_output.label_ids

    report = classification_report(
        labels,
        preds,
        target_names=[id2label[i] for i in range(3)],
        digits=4,
        zero_division=0
    )
    cm = confusion_matrix(labels, preds)

    print("\n===== Classification Report =====")
    print(report)

    print("\n===== Confusion Matrix =====")
    print(cm)

    # 9. save outputs
    with open(os.path.join(output_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
        f.write(report)

    np.savetxt(
        os.path.join(output_dir, "confusion_matrix.txt"),
        cm,
        fmt="%d"
    )

    result_df = test_df.copy()
    result_df["pred"] = preds
    result_df["pred_label"] = result_df["pred"].map(id2label)
    result_df["true_label"] = result_df["label"].map(id2label)
    result_df.to_csv(os.path.join(output_dir, "test_predictions.csv"), index=False, encoding="utf-8")

    with open(os.path.join(output_dir, "metrics_summary.txt"), "w", encoding="utf-8") as f:
        f.write("Validation:\n")
        f.write(str(val_result) + "\n\n")
        f.write("Test:\n")
        f.write(str(test_result) + "\n")




In [3]:
train_one_dataset(
    train_path="/kaggle/input/datasets/eddylu1792/tweeteval-data/Tweeteval_data/train.csv",
    val_path="/kaggle/input/datasets/eddylu1792/tweeteval-data/Tweeteval_data/val.csv",
    test_path="/kaggle/input/datasets/eddylu1792/tweeteval-data/Tweeteval_data/test.csv",
    text_column="comment",
    output_dir="outputs/hatebert_tweeteval",
    label_column="label",
    max_length=128,
    learning_rate=2e-5,
    train_batch_size=16,
    eval_batch_size=32,
    epochs=5
)

开始训练

===== Data Summary =====
Train size: 20909
label
0    13185
1     3941
2     3783
Name: count, dtype: int64

Val size: 2323
label
0    1437
1     459
2     427
Name: count, dtype: int64

Test size: 3830
label
0    2338
1     240
2    1252
Name: count, dtype: int64


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/151 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

{'eval_loss': 1.103958249092102, 'eval_accuracy': 0.7705553164012053, 'eval_macro_f1': 0.7425197694304261, 'eval_macro_precision': 0.7281090093779413, 'eval_macro_recall': 0.7612965215290858, 'eval_runtime': 7.5205, 'eval_samples_per_second': 308.888, 'eval_steps_per_second': 4.92, 'epoch': 3.0}

===== Test Result =====


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 2.6970064640045166, 'eval_accuracy': 0.5921671018276763, 'eval_macro_f1': 0.5202416431085374, 'eval_macro_precision': 0.56015025704495, 'eval_macro_recall': 0.5745816711321007, 'eval_runtime': 12.2673, 'eval_samples_per_second': 312.213, 'eval_steps_per_second': 4.891, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



===== Classification Report =====
              precision    recall  f1-score   support

      normal     0.8404    0.4234    0.5631      2338
   offensive     0.3444    0.3458    0.3451       240
  hatespeech     0.4956    0.9545    0.6525      1252

    accuracy                         0.5922      3830
   macro avg     0.5602    0.5746    0.5202      3830
weighted avg     0.6966    0.5922    0.5787      3830


===== Confusion Matrix =====
[[ 990  140 1208]
 [ 149   83    8]
 [  39   18 1195]]
